<a href="https://colab.research.google.com/github/renaljono249-stack/rtm1-2501010148/blob/main/akuisisi_openmeteo_labuanbajo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openmeteo-requests retry-requests requests-cache

import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": -8.4964,
	"longitude": 119.8877,
	"hourly": ["temperature_2m", "precipitation", "cloud_cover", "wind_speed_10m"],
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {abs(response.Latitude())}°{'S' if response.Latitude() < 0 else 'N'} {abs(response.Longitude())}°{'W' if response.Longitude() < 0 else 'E'}")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(1).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(2).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(3).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["precipitation"] = hourly_precipitation
hourly_data["cloud_cover"] = hourly_cloud_cover
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 890.7/890.7 kB 6.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 21.0 MB/s  0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.12.19
    Uninstalling flatbuffers-25.12.19:
      Successfully uninstalled flatbuffers-25.12.19
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [openmeteo-requests]
Coordinates: 8.471001625061035°S 119.92267608642578°E
Elevation: 58.0 m asl
Timezone difference to GMT+0: 0s

Hourly data
                          date  temperature_2m  precipitation  cloud_cover  \
0   2026-11-15 00:00:00+00:00       30.250000            0.0         40.0   
1   2026-11-15 01:00:00+00:00       31.700001            0.0         89.0   
2   2026-11-15 02:00:00+00:00       32.250000            0.0         90.0   
3   2026-11-15 03:00:00+00:00       31.549999            0.3         88.0   
4   2026-11-15 04:00:00+00:00       30.200001            0.7         95.0   
..          

In [ ]:
# Menghubungkan variabel hourly_dataframe ke df
df = hourly_dataframe.copy()

# Menambahkan nama lokasi
df.insert(0, "lokasi", "Labuan Bajo")

# Mengubah waktu UTC menjadi WITA
df["date"] = pd.to_datetime(df["date"], utc=True)

df["date"] = df["date"].dt.tz_convert("Asia/Makassar")

# Mengganti nama kolom menjadi bahasa Indonesia
df = df.rename(columns={
    "date": "waktu_wita",
    "temperature_2m": "suhu_c",
    "precipitation": "curah_hujan_mm",
    "cloud_cover": "tutupan_awan_persen",
    "wind_speed_10m": "kecepatan_angin_kmh"
})

# Menampilkan hasil
print("Jumlah record:", len(df))
print("Jumlah kolom:", len(df.columns))

display(df.head(10))

Jumlah record: 168
Jumlah kolom: 6


,lokasi,waktu_wita,suhu_c,curah_hujan_mm,tutupan_awan_persen,kecepatan_angin_kmh
0,Labuan Bajo,2026-11-15 08:00:00+08:00,30.250000,0.0,40.0,3.259938
1,Labuan Bajo,2026-11-15 09:00:00+08:00,31.700001,0.0,89.0,2.052316
2,Labuan Bajo,2026-11-15 10:00:00+08:00,32.250000,0.0,90.0,6.489992
3,Labuan Bajo,2026-11-15 11:00:00+08:00,31.549999,0.3,88.0,11.556504
4,Labuan Bajo,2026-11-15 12:00:00+08:00,30.200001,0.7,95.0,8.027678
5,Labuan Bajo,2026-11-15 13:00:00+08:00,31.049999,0.5,96.0,2.189795
6,Labuan Bajo,2026-11-15 14:00:00+08:00,31.250000,0.3,98.0,4.896529
7,Labuan Bajo,2026-11-15 15:00:00+08:00,31.100000,0.4,82.0,4.557236
8,Labuan Bajo,2026-11-15 16:00:00+08:00,30.400000,0.2,42.0,4.942631
9,Labuan Bajo,2026-11-15 17:00:00+08:00,29.450001,0.2,34.0,4.323748


In [ ]:

import requests
import pandas as pd
import json
import os
import time
from datetime import datetime

# 1. ENAM LOKASI WISATA
lokasi = [
    {"nama": "Labuan Bajo", "lat": -8.48043, "lon": 119.88839},
    {"nama": "Pulau Kelor", "lat": -8.52000, "lon": 119.79000},
    {"nama": "Pulau Padar", "lat": -8.64567, "lon": 119.57043},
    {"nama": "Pulau Komodo", "lat": -8.66857, "lon": 119.44372},
    {"nama": "Pulau Rinca", "lat": -8.66111, "lon": 119.66784},
    {"nama": "Pink Beach", "lat": -8.60116, "lon": 119.51977}
]

# 2. PARAMETER API
url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": ",".join(str(x["lat"]) for x in lokasi),
    "longitude": ",".join(str(x["lon"]) for x in lokasi),
    "hourly": "temperature_2m,precipitation,cloud_cover,wind_speed_10m",
    "timezone": "Asia/Makassar",
    "forecast_days": 7,
    "wind_speed_unit": "kmh"
}

headers = {
    "User-Agent": "Mahasiswa-INSTIKI/1.0 (Email: EMAIL_KAMPUS_KAMU)"
}

# 3. REQUEST API DENGAN RETRY
for percobaan in range(3):
    try:
        response = requests.get(
            url,
            params=params,
            headers=headers,
            timeout=30
        )
        response.raise_for_status()
        hasil = response.json()
        print("Data enam lokasi berhasil diambil!")
        break

    except requests.exceptions.RequestException as e:
        print("Error:", e)
        if percobaan == 2:
            raise
        time.sleep(2 ** (percobaan + 1))

# 4. MEMBUAT FOLDER
os.makedirs("data/raw", exist_ok=True)

# 5. MENYIMPAN JSON MENTAH
tanggal = datetime.now().strftime("%Y%m%d")

nama_json = f"data/raw/openmeteo_labuanbajo_{tanggal}.json"

with open(nama_json, "w", encoding="utf-8") as f:
    json.dump(hasil, f, ensure_ascii=False, indent=4)

# 6. MEMBENTUK DATAFRAME
if isinstance(hasil, dict):
    hasil = [hasil]

if len(hasil) != len(lokasi):
    raise ValueError("Jumlah lokasi tidak sesuai!")

semua_data = []

for tempat, data in zip(lokasi, hasil):

    hourly = data["hourly"]

    for i in range(len(hourly["time"])):

        semua_data.append({
            "lokasi": tempat["nama"],
            "waktu_wita": hourly["time"][i],
            "suhu_c": hourly["temperature_2m"][i],
            "curah_hujan_mm": hourly["precipitation"][i],
            "tutupan_awan_persen": hourly["cloud_cover"][i],
            "kecepatan_angin_kmh": hourly["wind_speed_10m"][i]
        })

df_multi = pd.DataFrame(semua_data)

# 7. MENYIMPAN CSV
nama_csv = f"data/raw/openmeteo_labuanbajo_{tanggal}.csv"

df_multi.to_csv(nama_csv, index=False)

# 8. PEMERIKSAAN HASIL
print("Jumlah lokasi:", df_multi["lokasi"].nunique())
print("Jumlah record:", len(df_multi))
print("Jumlah kolom:", len(df_multi.columns))

print("\nJumlah data setiap lokasi:")
print(df_multi.groupby("lokasi").size())

display(df_multi.head(10))

print("\nFile tersimpan:")
print(nama_json)
print(nama_csv)


Data enam lokasi berhasil diambil!
Jumlah lokasi: 6
Jumlah record: 1008
Jumlah kolom: 6

Jumlah data setiap lokasi:
lokasi
Labuan Bajo     168
Pink Beach      168
Pulau Kelor     168
Pulau Komodo    168
Pulau Padar     168
Pulau Rinca     168
dtype: int64


,lokasi,waktu_wita,suhu_c,curah_hujan_mm,tutupan_awan_persen,kecepatan_angin_kmh
0,Labuan Bajo,2026-11-15T00:00,25.7,0.0,26,4.3
1,Labuan Bajo,2026-11-15T01:00,25.4,0.0,25,3.7
2,Labuan Bajo,2026-11-15T02:00,25.1,0.0,41,6.9
3,Labuan Bajo,2026-11-15T03:00,24.9,0.0,51,7.6
4,Labuan Bajo,2026-11-15T04:00,24.8,0.0,67,9.3
5,Labuan Bajo,2026-11-15T05:00,24.7,0.0,41,7.9
6,Labuan Bajo,2026-11-15T06:00,25.1,0.0,24,8.6
7,Labuan Bajo,2026-11-15T07:00,27.7,0.0,11,7.8
8,Labuan Bajo,2026-11-15T08:00,30.3,0.0,40,3.3
9,Labuan Bajo,2026-11-15T09:00,31.7,0.0,89,2.1



File tersimpan:
data/raw/openmeteo_labuanbajo_20261115.json
data/raw/openmeteo_labuanbajo_20261115.csv


In [ ]:
import pandas as pd
import os
from datetime import datetime

# Mendapatkan tanggal dinamis hari ini untuk menemukan file yang benar
tanggal = datetime.now().strftime("%Y%m%d")
file_path = f"data/raw/openmeteo_labuanbajo_{tanggal}.csv"

# Membaca CSV Open-Meteo yang baru saja dibuat
if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(f"Berhasil memuat berkas: {file_path}")
else:
    # Fallback ke dataset multi yang sudah ada di memori jika file fisik belum tersimpan
    df = df_multi.copy()
    print("Memuat data langsung dari variabel df_multi di memori.")

# Mengubah format waktu WITA jika masih berupa string
df["waktu_wita"] = pd.to_datetime(df["waktu_wita"])

# Menampilkan hasil
display(df.head(10))

# Memeriksa jumlah data
print("Jumlah record:", len(df))
print("Jumlah kolom:", len(df.columns))

# Menyimpan hasil normalisasi ke folder processed
os.makedirs("data/processed", exist_ok=True)
df.to_csv(
    "data/processed/cuaca_labuan_bajo.csv",
    index=False
)
print("File hasil normalisasi berhasil disimpan di: data/processed/cuaca_labuan_bajo.csv")

Berhasil memuat berkas: data/raw/openmeteo_labuanbajo_20261115.csv


,lokasi,waktu_wita,suhu_c,curah_hujan_mm,tutupan_awan_persen,kecepatan_angin_kmh
0,Labuan Bajo,2026-11-15 00:00:00,25.7,0.0,26,4.3
1,Labuan Bajo,2026-11-15 01:00:00,25.4,0.0,25,3.7
2,Labuan Bajo,2026-11-15 02:00:00,25.1,0.0,41,6.9
3,Labuan Bajo,2026-11-15 03:00:00,24.9,0.0,51,7.6
4,Labuan Bajo,2026-11-15 04:00:00,24.8,0.0,67,9.3
5,Labuan Bajo,2026-11-15 05:00:00,24.7,0.0,41,7.9
6,Labuan Bajo,2026-11-15 06:00:00,25.1,0.0,24,8.6
7,Labuan Bajo,2026-11-15 07:00:00,27.7,0.0,11,7.8
8,Labuan Bajo,2026-11-15 08:00:00,30.3,0.0,40,3.3
9,Labuan Bajo,2026-11-15 09:00:00,31.7,0.0,89,2.1


Jumlah record: 1008
Jumlah kolom: 6
File hasil normalisasi berhasil disimpan di: data/processed/cuaca_labuan_bajo.csv
